In [1]:
import pandas as pd
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
data = pd.read_csv("dataset.csv").drop(columns=["Unnamed: 0"], errors="ignore")

data.head()

,source_text,plagiarized_text,label
0,Researchers have discovered a new species of b...,Scientists have found a previously unknown but...,1
1,The moon orbits the Earth in approximately 27....,Our natural satellite takes around 27.3 days t...,1
2,Water is composed of two hydrogen atoms and on...,H2O consists of 2 hydrogen atoms and 1 oxygen ...,1
3,The history of Rome dates back to 753 BC.,Rome has a long history that can be traced bac...,1
4,Pluto was once considered the ninth planet in ...,"In the past, Pluto was classified as the ninth...",1


In [3]:
print("Number of rows:", len(data))
print("Columns:", data.columns.tolist())

print("\nLabel count:")
print(data["label"].value_counts())

Number of rows: 370
Columns: ['source_text', 'plagiarized_text', 'label']

Label count:
label
0    187
1    183
Name: count, dtype: int64


In [4]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text

data["source_text"] = data["source_text"].apply(clean_text)
data["plagiarized_text"] = data["plagiarized_text"].apply(clean_text)

data.head()

,source_text,plagiarized_text,label
0,researchers have discovered a new species of b...,scientists have found a previously unknown but...,1
1,the moon orbits the earth in approximately 273...,our natural satellite takes around 273 days to...,1
2,water is composed of two hydrogen atoms and on...,h2o consists of 2 hydrogen atoms and 1 oxygen ...,1
3,the history of rome dates back to 753 bc,rome has a long history that can be traced bac...,1
4,pluto was once considered the ninth planet in ...,in the past pluto was classified as the ninth ...,1


In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def extract_features(source_text, submitted_text):
    """Turn a (source, submitted) pair into comparison features."""
    emb = embedder.encode([source_text, submitted_text])
    similarity = cosine_similarity([emb[0]], [emb[1]])[0][0]

    source_words = set(source_text.split())
    submitted_words = set(submitted_text.split())
    union = source_words | submitted_words
    word_overlap = len(source_words & submitted_words) / len(union) if union else 0.0

    len_diff = abs(len(source_words) - len(submitted_words))

    return [similarity, word_overlap, len_diff]

feature_rows = [
    extract_features(row["source_text"], row["plagiarized_text"])
    for _, row in data.iterrows()
]

features = pd.DataFrame(
    feature_rows,
    columns=["similarity", "word_overlap", "len_diff"]
)

X = features
y = data["label"]

features.head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,similarity,word_overlap,len_diff
0,0.938025,0.411765,0
1,0.657661,0.111111,4
2,0.802799,0.400000,1
3,0.928751,0.375000,4
4,0.944835,0.500000,2


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 296
Testing data: 74


In [7]:
from sklearn.naive_bayes import GaussianNB

#Logistic Regression
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)
prediction_lr = model_lr.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, prediction_lr))
print()
print(classification_report(y_test, prediction_lr))

Logistic Regression Accuracy: 0.7567567567567568

              precision    recall  f1-score   support

           0       0.70      0.89      0.79        37
           1       0.85      0.62      0.72        37

    accuracy                           0.76        74
   macro avg       0.78      0.76      0.75        74
weighted avg       0.78      0.76      0.75        74



In [8]:
#Random Forest
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train, y_train)
prediction_rf = model_rf.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, prediction_rf))
print()
print(classification_report(y_test, prediction_rf))

Random Forest Accuracy: 0.6891891891891891

              precision    recall  f1-score   support

           0       0.69      0.68      0.68        37
           1       0.68      0.70      0.69        37

    accuracy                           0.69        74
   macro avg       0.69      0.69      0.69        74
weighted avg       0.69      0.69      0.69        74



In [9]:
#Naive Bayes (Gaussian, since features are continuous, not counts)
model_nb = GaussianNB()
model_nb.fit(X_train, y_train)
prediction_nb = model_nb.predict(X_test)
print("Naive Bayes Accuracy:", accuracy_score(y_test, prediction_nb))
print()
print(classification_report(y_test, prediction_nb))

Naive Bayes Accuracy: 0.6756756756756757

              precision    recall  f1-score   support

           0       0.70      0.62      0.66        37
           1       0.66      0.73      0.69        37

    accuracy                           0.68        74
   macro avg       0.68      0.68      0.67        74
weighted avg       0.68      0.68      0.67        74



In [10]:
#Support Vector Machine
model_svm = SVC(kernel="linear", probability=True, random_state=42)
model_svm.fit(X_train, y_train)
prediction_svm = model_svm.predict(X_test)
print("SVM Accuracy:", accuracy_score(y_test, prediction_svm))
print()
print(classification_report(y_test, prediction_svm))

SVM Accuracy: 0.7432432432432432

              precision    recall  f1-score   support

           0       0.69      0.89      0.78        37
           1       0.85      0.59      0.70        37

    accuracy                           0.74        74
   macro avg       0.77      0.74      0.74        74
weighted avg       0.77      0.74      0.74        74



In [11]:
#Comparing models
accuracy_lr = accuracy_score(y_test, prediction_lr)
accuracy_rf = accuracy_score(y_test, prediction_rf)
accuracy_nb = accuracy_score(y_test, prediction_nb)
accuracy_svm = accuracy_score(y_test, prediction_svm)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Naive Bayes", "SVM"],
    "Accuracy": [accuracy_lr, accuracy_rf, accuracy_nb, accuracy_svm]
})
results = results.sort_values("Accuracy", ascending=False)
results

,Model,Accuracy
0,Logistic Regression,0.756757
3,SVM,0.743243
1,Random Forest,0.689189
2,Naive Bayes,0.675676


In [12]:
# Selecting best model
best_model_name = results.iloc[0]["Model"]

if best_model_name == "Logistic Regression":
    best_model = model_lr
elif best_model_name == "Random Forest":
    best_model = model_rf
elif best_model_name == "Naive Bayes":
    best_model = model_nb
else:
    best_model = model_svm

print("Best Model:", best_model_name)
print("Accuracy:", results.iloc[0]["Accuracy"])

Best Model: Logistic Regression
Accuracy: 0.7567567567567568


In [13]:
#Plagiarism detection (semantic-similarity features)
'''def check_plagiarism(source_text, submitted_text):
    source_text = clean_text(source_text)
    submitted_text = clean_text(submitted_text)

    feats = pd.DataFrame(
        [extract_features(source_text, submitted_text)],
        columns=["similarity", "word_overlap", "len_diff"]
    )

    prediction = best_model.predict(feats)[0]
    similarity = feats.iloc[0]["similarity"]

    result = "Plagiarism Detected" if prediction == 1 else "No Plagiarism Detected"

    print("Result:", result)
    print("Similarity:", round(similarity * 100, 2), "%")'''
def check_plagiarism(source_text, submitted_text):
    source_text = clean_text(source_text)
    submitted_text = clean_text(submitted_text)
    
    feats = pd.DataFrame([extract_features(source_text, submitted_text)], 
                         columns=["similarity", "word_overlap", "len_diff"])
    
    prediction = best_model.predict(feats)[0]
    similarity = feats.iloc[0]["similarity"]
    
    # Flag plagiarism if model predicts 1 OR if semantic similarity is over 60%
    if prediction == 1 or similarity >= 0.60:
        result = "Plagiarism Detected"
    else:
        result = "No Plagiarism Detected"
        
    print("Result:", result)
    print("Similarity:", round(similarity * 100, 2), "%")

In [14]:
source = "Researchers have discovered a new species of butterfly in the Amazon rainforest."
submitted = "Scientists have found a previously unknown species of butterfly in the Amazon rainforest."
check_plagiarism(source, submitted)

Result: Plagiarism Detected
Similarity: 97.21 %


In [15]:
source = "Researchers have discovered a new species of butterfly in the Amazon rainforest."
submitted = "Playing musical instruments enhances creativity."
check_plagiarism(source, submitted)

Result: No Plagiarism Detected
Similarity: 9.32 %


In [16]:
source = "Volunteering fosters community spirit."
submitted = "Volunteering fosters community spirit."
check_plagiarism(source, submitted)

Result: Plagiarism Detected
Similarity: 100.0 %


In [17]:
source = "I have done Msc. Data Science from MIT ACSC."
submitted = "I've completed MSc. data science from mit college."
check_plagiarism(source, submitted)

Result: Plagiarism Detected
Similarity: 86.4 %


In [18]:
import pickle

pickle.dump(best_model, open("plagiarismmodel.pkl", "wb"))
with open("embedder_name.txt", "w") as f:
    f.write("all-MiniLM-L6-v2")

print("Model saved successfully.")

Model saved successfully.
